# OS Tutor v2: second QLoRA fine-tuning round (Llama 3.1 8B)

What changed from the first round: 1580 training examples, about a fifth of them rewritten the way students type
(short, informal, typos: "sir pls explain srtf", "whats the wt of p2??") with the same engine-verified answers;
**2 epochs** instead of 1; a **checkpoint is saved to Google Drive** every ~10 minutes, so a disconnect loses at most
a few minutes; the model is exported as **os-tutor-v2.gguf**, so the first model stays as it is.

**Colab (works from a phone too):** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all** and allow
Google Drive access. Everything is saved in *My Drive → os-tutor-v2*.
**If the session disconnects:** open the notebook again and **Run all**. Training continues from the last checkpoint.

Takes about 2–2½ hours on a T4 (training ~1h45, export ~20 min). Then on the laptop: save
`os-tutor-v2.gguf` as `qlora\os-tutor-v2.gguf` and run `python -m qlora.create_finetuned --name os-tutor-v2`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q unsloth

## 1. Google Drive (checkpoints and the finished model go there)

In [ ]:
# Where everything is kept: Google Drive on Colab (survives a disconnect), the output folder on Kaggle
import os, pathlib
if os.path.exists("/kaggle/working"):
    WORK = "/kaggle/working"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/os-tutor-v2"
pathlib.Path(WORK, "checkpoints").mkdir(parents=True, exist_ok=True)
print("saving to", WORK)

## 2. Training data v2 (built in; a `data/train.jsonl` + `data/val.jsonl` in the Drive folder replaces it)

In [ ]:
import base64, gzip, json
TRAIN_FILE, VAL_FILE = f"{WORK}/data/train.jsonl", f"{WORK}/data/val.jsonl"
if os.path.exists(TRAIN_FILE) and os.path.exists(VAL_FILE):
    print("using the data files in", WORK + "/data")
else:
    DATA = "..."
    raw = gzip.decompress(base64.b64decode(DATA)).decode("utf-8")
    train_txt, val_txt = raw.split("\n=====SPLIT=====\n")
    pathlib.Path("data").mkdir(exist_ok=True)
    TRAIN_FILE, VAL_FILE = "data/train.jsonl", "data/val.jsonl"
    open(TRAIN_FILE, "w", encoding="utf-8").write(train_txt)
    open(VAL_FILE, "w", encoding="utf-8").write(val_txt)
print(sum(1 for _ in open(TRAIN_FILE, encoding="utf-8")), "training examples,",
      sum(1 for _ in open(VAL_FILE, encoding="utf-8")), "validation examples")

## 3. Settings

In [ ]:
import torch
EPOCHS = 2
MAX_SEQ = 2048
LORA_R = 16
LR = 2e-4
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH, ACCUM = (2, 4) if gpu_gb > 12 else (1, 8)      # effective batch 8 either way
print(f"GPU memory {gpu_gb:.0f} GB -> batch {BATCH} x accumulation {ACCUM}")

## 4. Load Llama 3.1 8B in 4-bit and add LoRA adapters

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", max_seq_length=MAX_SEQ, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
model.print_trainable_parameters()

## 5. Train (loss only on the tutor's replies; resumes from Drive if a checkpoint exists)

In [ ]:
import glob, inspect
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

ds = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VAL_FILE})
def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}
ds = ds.map(to_text, batched=True, remove_columns=ds["train"].column_names)

cfg = dict(dataset_text_field="text", per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
           warmup_steps=10, num_train_epochs=EPOCHS, learning_rate=LR, logging_steps=10, optim="adamw_8bit",
           weight_decay=0.01, lr_scheduler_type="linear", seed=3407, bf16=is_bfloat16_supported(),
           fp16=not is_bfloat16_supported(), output_dir=f"{WORK}/checkpoints", report_to="none",
           save_strategy="steps", save_steps=40, save_total_limit=2)          # a checkpoint every ~10 minutes
names = inspect.signature(SFTConfig.__init__).parameters
cfg["max_seq_length" if "max_seq_length" in names else "max_length"] = MAX_SEQ
common = dict(model=model, train_dataset=ds["train"], eval_dataset=ds["validation"], args=SFTConfig(**cfg))
try:
    trainer = SFTTrainer(tokenizer=tokenizer, **common)
except TypeError:
    trainer = SFTTrainer(processing_class=tokenizer, **common)
trainer = train_on_responses_only(trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n")
resume = bool(glob.glob(f"{WORK}/checkpoints/checkpoint-*"))
print("Resuming from the last checkpoint in Drive" if resume else "Starting a new training run")
stats = trainer.train(resume_from_checkpoint=resume or None)
print(stats)
print(trainer.evaluate())

## 6. Try it, including a student-style question

In [ ]:
FastLanguageModel.for_inference(model)
SYSTEM = "You are OS Tutor, a friendly tutor for the VIT BCSE303 Operating Systems course. You teach CPU scheduling and process synchronization (critical sections, semaphores, classic problems, deadlocks, the Banker's algorithm). Explain from scratch: an everyday example first, then the exact rule. Follow the class-notes conventions: ties go to the earlier arrival; a lower priority number means higher priority unless the question says otherwise; in Round Robin a process arriving when a slice ends joins the queue before the preempted one; TAT = CT \u2212 AT, WT = TAT \u2212 BT, RT = first start \u2212 AT. When a student shows you their answer, find the first mistake and give a hint before giving the answer away. Keep replies focused, and end with one short question or next step unless the student only wanted a fact."
for q in ["Explain SRTF to me like I'm new to it",
          "In round robin, if a new process arrives exactly when the quantum ends, who goes first?",
          "Quiz me on deadlocks", "sir whats the diff between sjf n srtf pls"]:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=350, temperature=0.3, do_sample=True)
    print("\n>>>", q, "\n" + tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## 7. Save the adapter to Drive (small backup, ~160 MB)

In [ ]:
import shutil
model.save_pretrained("os-tutor-v2-lora")
tokenizer.save_pretrained("os-tutor-v2-lora")
shutil.make_archive(f"{WORK}/os-tutor-v2-lora", "zip", "os-tutor-v2-lora")
print("saved", f"{WORK}/os-tutor-v2-lora.zip")

## 8. Merge and export the model for Ollama (GGUF Q4_K_M, ~15–25 minutes)

In [ ]:
DO_GGUF = True
if DO_GGUF:
    model.save_pretrained_gguf("os-tutor-gguf", tokenizer, quantization_method="q4_k_m")
    print("export finished")

## 9. Copy the model file to Google Drive

In [ ]:
import os, shutil
def find_q4():
    hits = []
    for root, dirs, files in os.walk("."):
        dirs[:] = [d for d in dirs if d not in ("drive", "sample_data") and not d.startswith(".")]
        hits += [os.path.join(root, f) for f in files if f.lower().endswith(".gguf") and "q4_k_m" in f.lower()]
    return hits
hits = find_q4()
assert hits, "No Q4_K_M .gguf file found. Did the export cell above finish without an error?"
src = max(hits, key=os.path.getsize)
print("model file:", src, f"({os.path.getsize(src) / 1e9:.1f} GB)")

dest = f"{WORK}/os-tutor-v2.gguf"          # Drive on Colab, the output folder on Kaggle
shutil.copy(src, dest)
print("Copied to", dest)
print("On the laptop: save it as qlora\\os-tutor-v2.gguf, then run  python -m qlora.create_finetuned --name os-tutor-v2")